# 00 · Vet the use case: is fine-tuning the coach's embedding model worth doing?

*For learning. Nothing here runs in CI, nothing costs money, and **no model is called**: this notebook reads the course content,
the runs already logged in MLflow (experiment `coach-retrieval-lab`, the smoke and diagnostic runs from `retrieval.py`), and the
vetting written in [`vetting/assessment.md`](vetting/assessment.md). It touches no holdout lesson; the only scores it shows were
logged on the **dev** split, which is the split for looking.*

**What this notebook settles.** Before any training: the use case in one line, the three gates with the honest evidence, where it
sits on the value-by-doable map, the use case card with its blank lines, the cost sheet, and the **bars** the fine-tune must clear
(agreed now, so nobody moves them to fit the result). It applies the course's *Use case and fit* module: "Is this a good AI use
case?", "Write the use case down", "Functional and non-functional requirements", "A well-architected review", "Total cost of
ownership" and "Is it fit for purpose?".

**Said plainly first.** Today's coach does **not** retrieve with an embedding model. The page tells it which lesson the learner is
on and puts the lesson's text in the prompt. So the champion for the retrieval step is `bge-large` as its makers ship it (recall@5
0.84 on 150 Check questions, PR #170), and a fine-tune has to earn a place only where the lesson is *not* known: a follow-up typed
on the study list, the blueprint's coach, Claude's tools, the claude.ai tutor project. On this computer none of those has recorded
a typed question yet.

**How to work through it.** Write your guess in the cell before each result, then run the next cell; the guesses go into MLflow
beside the real numbers. The exercises have blanks (`"___"`); the answers are in `00_vet_answers.py`.

> **In plain words: MLflow, an experiment and a run.** MLflow is the program that keeps the lab's evidence; here it runs on this
> computer at `http://localhost:5050` (the **tracking server**). The **experiment** `coach-retrieval-lab` is the folder. Each **run**
> is one permanent record with its **params** (settings), **metrics** (numbers), **tags** (labels such as the git commit) and
> **artifacts** (files), and a description of what it was for. A run can hold **child runs**, one per setup. A cell that fails marks
> the run FAILED instead of leaving an empty FINISHED one. The course: *Experiments, runs and the lifecycle*.

In [1]:
import importlib
import json
import os
import subprocess
import sys
from pathlib import Path

os.environ.setdefault("MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR", "false")
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")

import mlflow
import pandas as pd
from IPython.display import Markdown, display

import retrieval as R
import tracking
from drill.content import build_data

FILL = os.environ.get("DRILL_FILL_ANSWERS") == "1"      # only for checking the notebook end to end
answers = importlib.import_module("00_vet_answers")
VETTING = Path("vetting")


def attempt(value, name):
    # Your answer; if it still has a blank, a reminder (or, when checking the notebook end to end, the answer file's).
    if "___" not in str(value):
        return value
    if FILL:
        return getattr(answers, name)
    print("Fill in the ___ first, then run this cell again.")
    return None


tracking.connect()
run = mlflow.start_run(run_name="00_vet · your guesses and the bars",
                       description="One pass through `00_vet.ipynb`: the use case in one line, your guesses beside the numbers "
                                   "already logged on the dev split, the gates, the card, the cost sheet and the pilot bars. "
                                   "No model is called; nothing is scored here.")
mlflow.set_tags({"kind": "coach_retrieval_lab", "lab.stage": "00_vet", "git_commit": tracking.git_commit()})
tracking.guard_run()   # a cell that raises ends the run as FAILED, not as an empty FINISHED shell
you = tracking.Guesses()
data = build_data()
corpus, pairs = R.build_corpus(data), R.build_pairs(data)
splits = R.freeze_splits(data)   # deals and writes data/splits.json when it is missing (stage 1 explains the deal); returns it unchanged when it is there
print(f"{len(data['lessons'])} lessons in {len(data['modules'])} modules; {len(corpus)} Learn paragraphs; {len(pairs)} question-to-lesson pairs")

211 lessons in 28 modules; 741 Learn paragraphs; 1677 question-to-lesson pairs


## 1. The problem, and the use case in one line (lesson: *Is this a good AI use case?*)

The lesson's rule: start from a problem, not from the technology, and write the use case as **"Help [who] [do what] so [a number]
improves."** Then check the task happens often enough to matter.

> **In plain words.** When the coach is asked something and nobody has told it which lesson the question is about, it has to
> *find* the lesson: that step is **retrieval**. An **embedding model** turns each text into a list of numbers so that texts with
> similar meaning land close together; retrieval picks the five Learn paragraphs closest to the question. The score is
> **recall@5**: did one of the right lesson's paragraphs make the top five? Yes or no, per question, averaged.

Where is the problem today? Here is how the coach is called: the page sends the lesson id along with the question, and the server
records it as an `asked` event. The lesson text is already in the prompt; there is nothing to find. The doors with **no** lesson are
the ones that matter, and here is how many typed questions they have recorded on this computer:

In [2]:
import sqlite3

db = Path(R.HERE).parents[2] / ".drill" / "drill.db"      # the app's local database, when this checkout has one
if db.exists():
    with sqlite3.connect(db) as conn:
        asked = conn.execute("select count(*) from events where kind = 'asked'").fetchone()[0]
    print(f"{db}: {asked} typed questions ('asked' events)")
else:
    asked = 0
    print(f"No local app database at {db}. The main checkout had 0 'asked' events when this was vetted (2026-11-08).")
mlflow.log_metric("asked_events_on_this_computer", asked)
print("Real typed learner questions available for the test:", asked, "-> an Unknown and the lab's biggest risk.")

No local app database at /Users/huydo/Developer/dd-retrieval/.drill/drill.db. The main checkout had 0 'asked' events when this was vetted (2026-11-08).
Real typed learner questions available for the test: 0 -> an Unknown and the lab's biggest risk.


So the one line has to be honest about where the value is:

> **Help the Daily Drill coach find the right lesson for a question typed in a learner's own words, when the page has not told it
> the lesson, so recall@5 on learner-worded questions improves.**

### Exercise 1: the number in the one line

Three numbers were suggested for the "so [a number] improves" part. Which one measures the *job* (and not traffic or cost)?
Pick one of: `"questions answered a day"`, `"recall@5 on learner-worded questions"`, `"model size in MB"`.

In [3]:
SUCCESS_MEASURE = "___"   # one of the three strings above

In [4]:
mine = attempt(SUCCESS_MEASURE, "SUCCESS_MEASURE")
if mine is not None:
    assert mine == "recall@5 on learner-worded questions", ("Not quite: questions a day is how often the task happens (that is the "
        "'often enough to matter' check, a different question), and size is a cost. The job is finding the lesson.")
    print("Right. It is measured per question, yes or no, so a pass rate with a Wilson range: the course's 'Margins near 0% and 100%'.")
    mlflow.log_param("one_line.success_measure", mine)

Right. It is measured per question, yes or no, so a pass rate with a Wilson range: the course's 'Margins near 0% and 100%'.


## 2. The three gates, with the evidence (lesson: *Is this a good AI use case?*)

Fail any one and the idea is shelved: **a fixed rule could do it**; **no past examples with right answers**; **a wrong answer causes
harm and nobody checks**.

### Gate 1: could a fixed rule do it? Open.

The fixed rule is a **word-count table** (the real term is TF-IDF): it scores a paragraph by how many of the question's words it
contains, weighting rare words more. No learning from pairs, no GPU, 0.05 ms a question. If it finds lessons as well as the
champion, the embedding model is not the problem, and a fine-tune of it cannot be the answer.

**Your guess first.** In the 20-question smoke run, how many of 20 dev questions did the word table find in its top five? And the
champion `bge-large`?

> **In plain words: the smoke run and the numbers it prints.** A **smoke run** (the real term: smoke test) is a quick pass over 20
> questions to check that every piece works together; it is not a result. **McNemar p** is the **p-value**: if the two were really
> equal, the chance of a split at least this lopsided among the questions only one of them found (*Comparing on the same examples*).
> The **bootstrap range** redraws the questions at random, with repeats, many times and keeps the middle 95 in 100 gaps; a **Wilson
> range** does the same job for one share. Both are **confidence intervals**: they catch the true number about 95 times in 100
> (*How many examples do you need?*, *Margins near 0% and 100%*).

In [5]:
you.guess("smoke_tfidf_found_of_20", None)    # e.g. 10
you.guess("smoke_bge_found_of_20", None)      # e.g. 15

No guess written down. The habit only works if you commit to one before you look.
No guess written down. The habit only works if you commit to one before you look.


In [6]:
smoke = tracking.find_run(tracking.EXPERIMENT, "smoke · retrieval.py")
children = mlflow.search_runs([smoke.info.experiment_id], filter_string=f"tags.mlflow.parentRunId = '{smoke.info.run_id}'")
found = {row["tags.mlflow.runName"].split(" · recall")[0]: (row["metrics.recall_at_5"], row["metrics.n"]) for _, row in children.iterrows()}
tf = found[R.TFIDF.name]; bge = found[R.CHAMPION.name]
you.reveal("smoke_tfidf_found_of_20", tf[0] * tf[1], as_percent=False)
you.reveal("smoke_bge_found_of_20", bge[0] * bge[1], as_percent=False)
cmp = json.load(open(mlflow.artifacts.download_artifacts(run_id=smoke.info.run_id, artifact_path="comparison.json")))
print(f"word table {tf[0]:.2f} vs champion {bge[0]:.2f} on {tf[1]:.0f} questions; only the word table found {cmp['only_first']}, "
      f"only the champion {cmp['only_second']}; McNemar p = {cmp['p_value']:.2f}; "
      f"bootstrap range of the gap {cmp['gap_low']:+.2f} to {cmp['gap_high']:+.2f}")

smoke_tfidf_found_of_20: it was 12 (no guess written down)
smoke_bge_found_of_20: it was 10 (no guess written down)
word table 0.60 vs champion 0.50 on 20 questions; only the word table found 2, only the champion 0; McNemar p = 0.50; bootstrap range of the gap -0.25 to +0.00


**Reading it.** On this tiny sample the free rule came out *ahead*. Two things stop that being a finding. 20 questions give a range
about 30 points wide, and the two only disagreed on a handful (p 0.5: a coin would do that). And it is the known flaw of the easy
test: Check questions are written from the Learn text, so they share its words, which is exactly what a word table is good at. The
lab's answer is the **honest test**: each dev and test question rewritten the way a learner would type it (`build_honest_test`),
scored in stage 2. *This gate stays open until then.* If the word table still wins on learner-worded questions, the lab stops here
with a finding written in `DECISION.md`.

### Gate 2: are there past examples with right answers? Passed, with a risk.

Every Check question, extra practice question and prepared chip has a known lesson, and the lesson's Learn paragraphs are the
right answer. The pairs by where the question came from, and by the frozen split:

In [7]:
table = pd.crosstab(pd.Series([p["source"] for p in pairs], name="source"),
                    pd.Series([splits[p["lesson"]] for p in pairs], name="split"))[list(R.SPLITS)]
table["all"] = table.sum(axis=1)
display(table)
amb = R.ambiguous_questions(data)
print(f"{len(amb)} question texts dropped for sitting under several lessons, e.g. {amb[0]['question']!r} under {len(amb[0]['lessons'])} lessons: "
      "on their own they have no single right lesson.")
print("Lessons by split:", R.split_counts(splits), "· frozen in data/splits.json with seed", R.SEED)

split,train,dev,test,holdout,all
source,,,,,
check,123,42,28,18,211
more,242,82,56,36,416
prepared,605,204,142,99,1050


10 question texts dropped for sitting under several lessons, e.g. 'when does this stop working?' under 151 lessons: on their own they have no single right lesson.
Lessons by split: {'train': 123, 'dev': 42, 'test': 28, 'holdout': 18} · frozen in data/splits.json with seed 7


**But not one of these is a real typed learner question** (section 1 counted 0). The Check and more questions borrow the lesson's
words; the prepared chips are tapped, not typed. The stand-in is model-made **paraphrases**, each marked `checked: false` until a
person reads it. Here are a few of the ones made so far (the cache grows as stages 1 and 2 run), with how many of each question's content
words still appear in its lesson (the **word overlap**: lower means a fairer, harder question):

In [8]:
cache = R.DATA / "paraphrases.json"
if cache.exists():
    store = json.loads(cache.read_text())
    texts = R.lesson_text(corpus)
    by_id = {p["id"]: p for p in pairs}
    rows = []
    for qid, entries in store.items():
        p = by_id.get(qid)
        if p:
            for e in entries:
                rows.append({"original": p["question"], "paraphrase": e["text"], "checked": e["checked"],
                             "overlap before": R.word_overlap(p["question"], texts[p["lesson"]]),
                             "overlap after": R.word_overlap(e["text"], texts[p["lesson"]])})
    para = pd.DataFrame(rows)
    pd.set_option("display.max_colwidth", 120)
    display(para.head(6))
    print(f"{len(para)} paraphrases so far, {int(para.checked.sum())} read by a person; "
          f"average word overlap {para['overlap before'].mean():.2f} before, {para['overlap after'].mean():.2f} after")
    mlflow.log_metric("paraphrases_so_far", len(para))
    mlflow.log_metric("paraphrases_checked_by_a_person", int(para.checked.sum()))
else:
    print("No paraphrases yet: stage 2 makes them.")

,original,paraphrase,checked,overlap before,overlap after
0,Agent A succeeds 9 times in 10 at $0.40 per success. Agent B succeeds 8 times in 10 at $0.05 per success. Which is b...,"If one person makes money 9 out of 10 times at 40 cents each, and another makes money 8 out of 10 times at 5 cents e...",False,0.364,0.000
1,Your report works out cost per success using only the bills of the runs that succeeded. What does it miss?,"If my report only looks at the bills from the runs that worked, what am I not seeing?",False,0.444,0.167
2,You want a model to grade whether each agent run finished its task. What should you do before trusting its grades?,Check if the model is actually good at judging whether the agents finished their tasks before relying on its judgments.,False,0.444,0.273
3,Why does pausing for a person to approve (human-in-the-loop) need a checkpointer?,Why do I need to save progress when waiting for someone to approve something?,False,0.400,0.167
4,"Your agent may issue a refund, but only after a person says yes. Where should it call interrupt() to pause?","If someone needs to say yes before getting a refund, where should the code wait for that yes?",False,0.571,0.000
5,"Your agent paused at 5pm, waiting for a manager. The yes arrives the next morning. What tells LangGraph which saved ...","If the agent stops at 5pm and then starts again the next day, what does LangGraph use to know which previous session...",False,0.200,0.250


208 paraphrases so far, 0 read by a person; average word overlap 0.54 before, 0.37 after


**Reading it.** The overlap drops, which is what we want. But read the Agent A one: "Agent A succeeds 9 times in 10" became "one person
makes money 9 out of 10 times", and the lesson is about agents, not people. A paraphrase that changes the meaning is a wrong test
case, and a model would be blamed for missing a lesson the question no longer asks about. That is why `checked` exists, and why the
quality bar below says *read by a person first*. The gate passes because the examples exist; the risk is that they are not how
learners ask. (Finding: the overlap numbers. Hunch: how far these differ from real questions. Wrong if the first real `asked`
events look just like the paraphrases.)

### Gate 3: does a wrong answer cause harm that nobody checks? Passed.

**Augment**, not automate: retrieval only chooses which paragraphs the coach reads; the coach writes an answer; the learner reads it
and taps **Got it** or **Still stuck**, which the app already records as an event. A wrong lesson means a confusing answer and a
Still stuck, which is also the signal stage 7 watches. Nothing is graded, decided or made irreversible by the retrieval.

### Exercise 2: which gate is still open?

In [9]:
GATE_OPEN = "___"   # "fixed rule", "no examples" or "unchecked harm" 

In [10]:
mine = attempt(GATE_OPEN, "GATE_OPEN")
if mine is not None:
    assert mine == "fixed rule", "Not quite: the examples exist (with a risk) and a person checks every answer. What is not yet shown is that a word table cannot do the job."
    print("Right. Stage 2 settles it on the learner-worded questions; until then the honest word is 'open', not 'passed'.")
    mlflow.log_param("gates.open", mine)

Right. Stage 2 settles it on the learner-worded questions; until then the honest word is 'open', not 'passed'.


## 3. The map: value by doable (lesson: *Is this a good AI use case?*)

Two axes, kept apart. **Value**: what the problem costs and how much the business cares. **Doable**: the data is ready, AI suits
the task, the team can change how it works. The corners: *accelerate* (both high), *research* (value only), *incubate* (doable only),
*shelve* (neither).

Before the call, the one number that says how big the problem is on the easy questions. **Your guess first:** on all 328 dev
questions, what share did the shipped champion find in its top five? And on the Check questions alone?

In [11]:
you.guess("dev_recall_all", None)      # e.g. 0.8
you.guess("dev_recall_check", None)    # e.g. 0.9

No guess written down. The habit only works if you commit to one before you look.
No guess written down. The habit only works if you commit to one before you look.


In [12]:
diag = tracking.find_run(tracking.EXPERIMENT, "bge-large · all dev")
rows = pd.DataFrame(json.load(open(mlflow.artifacts.download_artifacts(run_id=diag.info.run_id, artifact_path="rows.json"))))
chip_of = {p["id"]: p["chip"] for p in pairs}   # the question kind, read back from the pairs
rows["group"] = [s if s != "prepared" else "prepared · " + chip_of.get(i, "?") for s, i in zip(rows.source, rows.id)]
you.reveal("dev_recall_all", rows.hit.mean())
you.reveal("dev_recall_check", rows[rows.source == "check"].hit.mean())
groups = rows.groupby("group").hit.agg(found="sum", n="count")
groups[["recall@5", "low", "high"]] = [tracking.wilson(int(f), int(n)) for f, n in zip(groups.found, groups.n)]
display(groups.sort_values("recall@5", ascending=False).round(2))
print(f"Run: {tracking.run_link(diag)}")

dev_recall_all: it was 63.4% (no guess written down)
dev_recall_check: it was 88.1% (no guess written down)


,found,n,recall@5,low,high
group,,,,,
check,37,42,0.88,0.75,0.95
more,69,82,0.84,0.75,0.90
prepared · term,21,26,0.81,0.62,0.91
prepared · explain:gotcha,2,4,0.50,0.15,0.85
prepared · option,79,174,0.45,0.38,0.53


Run: http://localhost:5050/#/experiments/6 · run "bge-large · all dev · 328 questions" (f7ffae1d)


**Reading it.** The Check and more questions reproduce PR #170's 0.84; the whole-dev average is dragged to 0.63 by the **option
chips** ("Why not 'a timeout'?"), which quote a wrong answer and name no topic. They are tapped beside a question, never typed, so
they are not retrieval questions at all; the recommendation to stages 1 and 2 is to look at them apart or leave them out, and say
why. (This was a hunch when a 20-question sample scored 0.50; checking all 328 made it a finding. Wrong if learners turn out to
type option-shaped questions.) Look at the ranges, not the points: the term chips are 26 questions, so their range is wide.

**The call.** *Value for Daily Drill alone: low.* The main flow has no retrieval problem, and the doors that do have 0 recorded typed
questions here. *Doable: high.* The data is ready and frozen, the score is yes-or-no, the champion runs in 5 to 6 ms a question
and the whole course embeds in about 10 to 16 s on this computer, and the owner can change the coach. **Corner: incubate**, a safe
way to learn, which is exactly what the lab is for: the enterprise value is rehearsing the whole lifecycle (vet, data, baselines,
train, evaluate, register, serve, monitor) on a problem small enough to finish, then doing the same on Azure Databricks. To move to
*accelerate*, the owner needs recorded questions where the coach missed because it did not know the lesson.

### Exercise 3: place it

In [13]:
VALUE, DOABLE = "___", "___"   # each "high" or "low" 

In [14]:
value, doable = (attempt(VALUE, "VALUE"), attempt(DOABLE, "DOABLE"))
if value is not None and doable is not None:
    assert (value, doable) == ("low", "high"), ("Not quite: no cost of the problem is known and no typed questions are recorded (value low); "
        "the data, the measure and this computer's GPU are all ready (doable high).")
    corner = {("high", "high"): "accelerate", ("high", "low"): "research", ("low", "high"): "incubate", ("low", "low"): "shelve"}[(value, doable)]
    print(f"Right: {corner}. The lifecycle rehearsal is the value; Daily Drill's own gain waits for real traffic.")
    mlflow.log_params({"map.value": value, "map.doable": doable, "map.corner": corner})

Right: incubate. The lifecycle rehearsal is the value; Daily Drill's own gain waits for real traffic.


## 4. The use case card (lesson: *Write the use case down*)

Ten questions on one page, so everyone agrees on the same job and the trial knows what to measure. A blank line is an open
question to answer before building.

> **In plain words: the card's numbers.** **hit@1**: the closest paragraph is the right lesson's. **MRR** (mean reciprocal rank):
> each question scores 1 divided by the place of the first right paragraph (1 for first, a half for second, a third for third, 0 if
> never found), averaged over the questions; the course defines it in *Measuring retrieval*. The **fingerprint** is a short code
> worked out from every paragraph and question of the course, so it changes when any lesson changes; a dataset's **digest**
> (stage 1) is the same idea for one dataset's rows.

In [15]:
card = {
    "user": "The Daily Drill coach, on behalf of a learner who typed a question where no lesson is known",
    "task": "Put the right lesson's Learn paragraphs in front of the coach before it answers",
    "decision": "Which five of the 741 Learn paragraphs are closest in meaning to the typed question",
    "action": "The coach answers with those paragraphs as context; the learner taps Got it or Still stuck",
    "today": "UNKNOWN: where the lesson is known there is no retrieval; where it is not, the coach answers from the curriculum map alone; no typed questions recorded here",
    "success": "recall@5 on learner-worded questions, with hit@1, MRR and the weakest module beside it",
    "wrong_answer": "A confusing answer, caught by the learner and recorded as Still stuck; nothing graded or irreversible",
    "data": f"{len(pairs)} pairs, {len(corpus)} paragraphs, fingerprint {R.content_fingerprint(corpus, pairs)}; frozen lesson splits {R.split_counts(splits)}; paraphrases as the learner-worded test, checked: false until read. UNKNOWN: no real typed questions yet",
    "feedback": "asked events and Got it / Still stuck already recorded; stage 7 adds which lesson retrieval picked; real questions join the test as they arrive",
    "owner": "The lab's learner, who owns Daily Drill",
}
display(pd.DataFrame({"answer": card}))
blank = [k for k, v in card.items() if "UNKNOWN" in v]
print("Blank or part-blank lines:", blank)
mlflow.log_dict(card, "vetting/card.json")
mlflow.log_param("card.blank_fields", ",".join(blank));


,answer
user,"The Daily Drill coach, on behalf of a learner who typed a question where no lesson is known"
task,Put the right lesson's Learn paragraphs in front of the coach before it answers
decision,Which five of the 741 Learn paragraphs are closest in meaning to the typed question
action,The coach answers with those paragraphs as context; the learner taps Got it or Still stuck
today,"UNKNOWN: where the lesson is known there is no retrieval; where it is not, the coach answers from the curriculum map..."
success,"recall@5 on learner-worded questions, with hit@1, MRR and the weakest module beside it"
wrong_answer,"A confusing answer, caught by the learner and recorded as Still stuck; nothing graded or irreversible"
data,"1677 pairs, 741 paragraphs, fingerprint 33910f9634f6; frozen lesson splits {'train': 123, 'dev': 42, 'test': 28, 'ho..."
feedback,asked events and Got it / Still stuck already recorded; stage 7 adds which lesson retrieval picked; real questions j...
owner,"The lab's learner, who owns Daily Drill"


Blank or part-blank lines: ['today', 'data']


## 5. Requirements and the review (lessons: *Functional and non-functional requirements*, *A well-architected review*)

What it **must do**: given a question and no lesson, return the five closest paragraphs with their lesson ids; search the whole
course every time; log which lesson it picked with every answer; load the model by its registry alias so a rollback is an alias
move; never touch the holdout before the final, once-only score.

**How well**, every line with a number, and the six pillars with a risk each: both are written out in
[`vetting/assessment.md`](vetting/assessment.md), sections 6 and 7. The two pillars rated *medium* are **Cost** (on Databricks a
served custom model with scale-to-zero off runs about 730 hours a month whatever the traffic, against a pay-per-token champion with
no upkeep) and **Governance** (a test set nobody has read is not yet a test set). Everything else is low because the fallback is
today's behaviour and the model only ranks paragraphs.

## 6. The cost sheet (lesson: *Total cost of ownership*)

The skill's script adds up every line over the years the system runs and names the biggest item. Here every local line is $0 (Ollama,
the local MLflow server, a cached model; the owner's time counted in hours), and every Databricks line is **Unknown**: counted as 0
and listed with the question for the owner. Stage 8 fills them in from the workspace's prices; nothing is guessed.

In [16]:
tco = Path(R.HERE).parents[2] / ".claude" / "skills" / "vet-ai-use-case" / "scripts" / "tco.py"   # the repo root is two levels up from this lab folder
inputs = VETTING / "tco_inputs.json"
if tco.exists():
    out = subprocess.run([sys.executable, str(tco), str(inputs)], capture_output=True, text=True, check=True).stdout
    out += ("\n\n(The script prints only the lines above $0, and today there are none, so its table is empty. "
            "Every line, by hand, is in vetting/assessment.md section 8.)")
else:
    out = f"(scripts/tco.py was looked for at {tco} and is not there; the cost table, line by line, is in vetting/assessment.md section 8)"
display(Markdown(out))
unknowns = json.loads(inputs.read_text())["_unknowns"]
display(pd.DataFrame(unknowns))
mlflow.log_artifact(str(inputs), "vetting")
mlflow.log_text(out, "vetting/tco_table.md")
mlflow.log_param("tco.unknown_lines", len(unknowns));


| Item | Over 2 years |
|---|---|


**Total: $0**, of which $0 a month to run. Biggest item: none.


(The script prints only the lines above $0, and today there are none, so its table is empty. Every line, by hand, is in vetting/assessment.md section 8.)

,line,question
0,llm.questions_per_month,"How many coach questions a month arrive with NO lesson known (study list follow-ups, the blueprint's coach, Claude's..."
1,llm.embedding_tokens_per_month / usd_per_m_embedding,"If the champion stays a pay-per-token endpoint (databricks-bge-large-en), what is its price per million tokens in yo..."
2,retrieval.vector_search_hours / usd_per_vector_search_hour,"Will the lesson paragraphs live in a Databricks AI Search (formerly Vector Search) index, and at what hourly price a..."
3,databricks.usd_per_dbu,The DBU price for your workspace's region and tier (jobs compute and model serving are priced differently).
4,databricks.job_dbus_per_hour / job_hours_per_month,"The GPU job that fine-tunes (stage 8): DBUs per hour for the GPU type, hours per run, and how many runs a month once..."
5,databricks.serving_dbus_per_hour / serving_hours_per_month,A served custom embedding model with scale-to-zero OFF runs about 730 hours a month. DBUs per hour for the smallest ...
6,storage.usd_per_gb_month,"About 2 GB (1.34 GB of weights per model version, the pairs and the traces); the per-GB price is your region's."
7,people_monthly,"Who reads the model-made paraphrases before they count as the honest test, and who owns the custom model's upkeep at..."


> **In plain words: the serving words.** A **serving endpoint** is a web address that runs a model and answers requests.
> **Pay-per-token** means billed for each token used (the pieces of words a model reads: *How an LLM works, in plain words*), with
> nothing to keep running. A served custom model is billed in **DBUs** (the unit Databricks bills in) for every hour it is up;
> **scale-to-zero** shuts it down when idle, so it costs nothing then, but the first question waits while it starts. A **registry
> alias** is a name like `@champion` that points at one model version, so a rollback is moving the name (*Model registry and promotion*).

### Exercise 4: the line most likely to be the biggest

A served model with **scale-to-zero off** is billed for every hour of the month, whether or not anyone asks a question (the lesson
"Pay-per-token or provisioned throughput?" is the same trade-off for LLMs). About how many hours is that a month?

In [17]:
SERVING_HOURS = "___"   # hours in an average month

In [18]:
mine = attempt(SERVING_HOURS, "SERVING_HOURS")
if mine is not None:
    assert 720 <= float(mine) <= 745, "Not quite: 24 hours times 365 days, divided by 12 months."
    print(f"Right: about {24 * 365 / 12:.0f}. At low traffic that line dwarfs the per-question cost of a pay-per-token endpoint, "
          "which is why the champion's $0-upkeep line is the custom model's real competitor at work.")
    mlflow.log_param("tco.serving_hours_per_month", float(mine))

Right: about 730. At low traffic that line dwarfs the per-question cost of a pay-per-token endpoint, which is why the champion's $0-upkeep line is the custom model's real competitor at work.


## 7. The bars (lessons: *Is it fit for purpose?*, *A must-pass set*, *Comparing on the same examples*, *Margins near 0% and 100%*)

**Fit for purpose** means good enough for this job, measured on real examples, against lines agreed *before* the trial. Four of them,
each with its reason, never a round number for its own sake. The call after the pilot is go, fix or stop, read off these lines.

In [19]:
bars = pd.DataFrame([
    {"bar": "Quality",
     "line": "On the test lessons' learner-worded questions (paraphrases of the 84 Check and more questions in the test split, read and "
             "corrected by a person first), the fine-tune's recall@5 beats the champion with the paired bootstrap 95% range of the gap "
             "above zero, and McNemar on the disagreements agreeing.",
     "why": "The champion is already at 0.84 to 0.88 on the easy questions and a custom model adds upkeep a pay-per-token endpoint does "
            "not have, so a win must be unmistakable on the hard questions, not a point that could be luck."},
    {"bar": "Must-pass",
     "line": "No module in the test split worse than the champion by more than the bootstrap noise (its paired range includes zero or sits "
             "above it); the forgetting check, paired like the win: on the course-worded Check and practice questions scored for both models, "
             "the paired bootstrap range of (fine-tune minus champion) includes zero or sits above it and McNemar shows no champion win; all four "
             "leakage checks pass on the exact data scored; the holdout is scored once, after the model is chosen, and reported whatever it says.",
     "why": "An average can win while one module gets worse, and a model taught the course's wording must not lose what its makers taught it. "
            "Paired, because 'inside the champion's Wilson range' would wave through a model that lost nine questions the champion found and "
            "none the other way, which McNemar calls a real loss (p 0.004): that range measures the champion's uncertainty, not a tolerance."},
    {"bar": "Speed",
     "line": "Milliseconds per question no slower than the champion by more than the champion's own pass-to-pass spread, both measured in the "
             "same session on the same questions: the median of several passes, same batch size, model load off the clock (stage 2 measures "
             "the champion's spread this way and writes the numbers in its run; a single pass is not enough, it moved 4.5 to 6.6 ms between two sets).",
     "why": "A difference smaller than the pass-to-pass noise cannot be told from the champion; a bigger one is a real slowdown, and for a fine-tune "
            "of the same weights (1.34 GB) that would be a serving mistake, not a trade-off. If the base model changes (embeddinggemma), the line "
            "is re-measured for that base and this reason rewritten."},
    {"bar": "Cost",
     "line": "$0 on this computer. On Databricks, filled in at stage 8: the custom model's serving line against the pay-per-token champion's "
             "at the real no-lesson question volume, with the gain in questions found per month beside it.",
     "why": "The owner can then say whether each extra found question is worth its price; today every Databricks price is Unknown."},
])
display(Markdown("\n".join(f"- **{b.bar}:** {b.line}  \n  *Why:* {b.why}" for _, b in bars.iterrows())))
for _, b in bars.iterrows():
    mlflow.log_param(f"bar.{b.bar.lower()}", b.line[:500])
mlflow.log_artifact(str(VETTING / "assessment.md"), "vetting")

- **Quality:** On the test lessons' learner-worded questions (paraphrases of the 84 Check and more questions in the test split, read and corrected by a person first), the fine-tune's recall@5 beats the champion with the paired bootstrap 95% range of the gap above zero, and McNemar on the disagreements agreeing.  
  *Why:* The champion is already at 0.84 to 0.88 on the easy questions and a custom model adds upkeep a pay-per-token endpoint does not have, so a win must be unmistakable on the hard questions, not a point that could be luck.
- **Must-pass:** No module in the test split worse than the champion by more than the bootstrap noise (its paired range includes zero or sits above it); the forgetting check, paired like the win: on the course-worded Check and practice questions scored for both models, the paired bootstrap range of (fine-tune minus champion) includes zero or sits above it and McNemar shows no champion win; all four leakage checks pass on the exact data scored; the holdout is scored once, after the model is chosen, and reported whatever it says.  
  *Why:* An average can win while one module gets worse, and a model taught the course's wording must not lose what its makers taught it. Paired, because 'inside the champion's Wilson range' would wave through a model that lost nine questions the champion found and none the other way, which McNemar calls a real loss (p 0.004): that range measures the champion's uncertainty, not a tolerance.
- **Speed:** Milliseconds per question no slower than the champion by more than the champion's own pass-to-pass spread, both measured in the same session on the same questions: the median of several passes, same batch size, model load off the clock (stage 2 measures the champion's spread this way and writes the numbers in its run; a single pass is not enough, it moved 4.5 to 6.6 ms between two sets).  
  *Why:* A difference smaller than the pass-to-pass noise cannot be told from the champion; a bigger one is a real slowdown, and for a fine-tune of the same weights (1.34 GB) that would be a serving mistake, not a trade-off. If the base model changes (embeddinggemma), the line is re-measured for that base and this reason rewritten.
- **Cost:** $0 on this computer. On Databricks, filled in at stage 8: the custom model's serving line against the pay-per-token champion's at the real no-lesson question volume, with the gain in questions found per month beside it.  
  *Why:* The owner can then say whether each extra found question is worth its price; today every Databricks price is Unknown.

**What 84 questions can and cannot show.** The quality bar is a paired comparison: only the questions where champion and challenger
*disagree* count. The course's quick rule ("Comparing on the same examples"): 20 disagreements need a gap of about 9, 100 need
about 20. So if the two disagree on 20 of the 84 test questions, the fine-tune must win about 15 of them to 5. A real gain of a few
points will read as "not shown". That is the honest outcome of a small test, not a failure of the model, and it is why the bar is a
range above zero and not "0.90".

### Exercise 5: how big a gap?

Champion and fine-tune disagree on 20 of the test questions. By the quick rule, about how many more must the fine-tune win than
the champion for the win to be more than luck?

> **In plain words: power, and the square root.** The chance that a test notices a real gain of a given size is its **power** (the
> real term: statistical power, from *Enough to notice a difference*); 84 test questions give low power for a gain of a few points.
> The **square root** of a number is the number that, multiplied by itself, makes it: the square root of 25 is 5, and of 16
> is 4. The quick rule in that lesson uses the square root of the number of disagreements.

In [20]:
NEEDED_GAP = "___"   # a whole number of questions

In [21]:
mine = attempt(NEEDED_GAP, "NEEDED_GAP")
if mine is not None:
    assert 8 <= int(mine) <= 10, "Not quite: the lesson's rule of thumb for 20 disagreements is a gap of about 9 (roughly twice the square root of 20)."
    print(f"Right, about {int(mine)}: 15 to 5 is a real win, 12 to 8 is too close to call. "
          f"The course's exact check: McNemar p for 15 vs 5 = {tracking.mcnemar(15, 5):.3f}; for 12 vs 8 = {tracking.mcnemar(12, 8):.3f}.")
    mlflow.log_param("bars.quick_rule_gap_for_20_disagreements", int(mine))

Right, about 9: 15 to 5 is a real win, 12 to 8 is too close to call. The course's exact check: McNemar p for 15 vs 5 = 0.041; for 12 vs 8 = 0.503.


## 8. What happens next

- **Stage 1 (`01_pairs.ipynb`)** registers the pairs and the frozen splits as MLflow datasets with digests, and decides what to do
  with the option chips (apart, or out, with the reason written down).
- **Stage 2 (`02_baselines.ipynb`)** settles gate 1: the word table against the champion and `embeddinggemma`, on all dev questions
  and on the learner-worded ones, which it first makes and a person reads. If the word table wins there, the lab stops with a
  finding in `DECISION.md`. If the champion wins but only narrowly, the champion is the model and no fine-tune is needed. Only a
  clear gap between the word table and the champion says the embedding model is where the quality lives, and then the fine-tune
  has something to improve.
- The **bars above are frozen now**. If one is changed later, the change and the reason go in the run that uses it.

The vetting itself is [`vetting/assessment.md`](vetting/assessment.md), in the skill's format, with every Unknown as a question for the
owner.

In [22]:
mlflow.end_run()
print(f"This pass is saved: {tracking.run_link(run)}")

This pass is saved: http://localhost:5050/#/experiments/6 · run "00_vet · your guesses and the bars" (9375d55d)
